# Backbone Experiment: vit_b_16

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: { 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


/Users/christinachau/PycharmProjects/asfr/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## Shared Config

In [2]:
# Only line to change
BACKBONE = "vit_b_16"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.train.backbone_lr   = 1e-4  
    cfg.train.lr            = 1e-4
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug       = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [ ]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: mps


Training spatial-only baseline (vit_b_16) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.2973 | val_acc=89.1%


Epoch   2/50 | train_loss=0.2257 | val_acc=92.2%


Epoch   3/50 | train_loss=0.1994 | val_acc=92.0%


...

Epoch  29/50 | train_loss=0.0387 | val_acc=96.2%


Epoch  30/50 | train_loss=0.0375 | val_acc=96.2%


Epoch  31/50 | train_loss=0.0352 | val_acc=96.2%


Epoch  32/50 | train_loss=0.0307 | val_acc=96.3%


Epoch 33/50:  12%|█▏        | 163/1329 [03:14<15:57,  1.22batch/s]

## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [ ]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

In [ ]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch. `b`is the frequency weight.
If `b` drops below 0.1 early in training the frequency branch is being ignored.


In [ ]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

In [ ]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [ ]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

In [ ]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3, essentially teaching itself what the frequency branch provides.


In [ ]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

In [ ]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

## Summary: vit_b_16
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [ ]:
df = pd.read_csv("./results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
